In [ ]:
#inputs: prarie vole and mouse SAM
#outputs: mapped subclasses between prarie vole and mouse

In [4]:
sam1=SAM()
sam1.load_data('../../SAM_MO_soupx_cleaned_08262026.h5ad')

In [1]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 41.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [2]:
!pip install loompy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for loompy: filename=loompy-3.0.8-py3-none-any.whl size=54013 sha256=8a554b9fe418509e87c938711dcc034ec4c210a3cd210c292db6fc85ea2c70fd
  Stored in directory: /root/.cache/pip/wheels/7a/b0/51/0054e104762c74124646fb54c3748d0fddd8efb7d5d5514464
  Created wheel for numpy-groupies: filename=numpy_groupies-0.9.22-py3-none-any.whl size=25846 sha256=3ce8092bbd9987369c71d9ea2f7cdf47579a60a2ca63496a9bec671172ead730
  Stored in directory: /root/.cache/pip/wheels/2e/b9/5a/225e71b783e29f2098ba2dc8a5266f02b2d0d08f1890a28548
Successfully built loompy numpy-groupies


In [10]:
!pip install harmonypy=='0.0.9'

  Attempting uninstall: harmonypy
    Found existing installation: harmonypy 0.0.5
    Uninstalling harmonypy-0.0.5:
      Successfully uninstalled harmonypy-0.0.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import loompy
import time
import pickle

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test = sc.read_loom('../../subset_Allen_institute_Full_subclass_test_250_0.loom')

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [3]:
test.obs['library_method'].value_counts()

10Xv3    58575
10Xv2    19607
Name: library_method, dtype: int64

In [ ]:
for indexer in range(0,30):
    org = 'mo'
    ref = 'mg'
    dat = sc.read_loom('../../subset_Allen_institute_Full_subclass_test_250_'+str(i)+'.loom')
    dat.obs_names = dat.obs['obs_names']
    dat.var_names = list(dat.var['x'])
    
    sam=SAM(dat)
    sam.preprocess_data()
    sam.run(batch_key='library_method')
    
    #make sure that the obs and var names are unique
    sam.adata.obs_names_make_unique()
    sam.adata.var_names_make_unique()

    sam1.adata.obs_names_make_unique()
    sam1.adata.var_names_make_unique()
    
    sams = {ref:sam,org:sam1}

    sm = SAMAP(
        sams,
        f_maps = '../../BLASTMAPPING/Hypo_proj/mgmo/')
    sm.run(pairwise=True)
    save_samap(sm,'../../sm_Allen_Full_mo_soupx_cleaned_08262026_subclass_250_'+str(indexer)+'.pkl')
    
    level = 'subclass_id_label'
    t0 = time.time()
    spline = []
    num_cell_types = []
    best_reso = ###manually set best resolution
    r = range(best_reso,best_reso + 5,5)
    

    #range of leiden clustering resolutions 
    for reso in r:
        sm.sams[org].clustering(param=reso)

        #get mapping between mouse and org
        keys = {ref:level,org:'leiden_clusters'}
        D,MappingTable = get_mapping_scores(sm,keys)
        lim_MappingTable = MappingTable.filter(like=org + '_')
        lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

        #find the best mapping that has more than 25 cells and greater than .2 alignment score 
        mapping_dict = {}
        for item in lim_MappingTable:
            len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['leiden_clusters'] == int(item[3:])])
            if len_item > 25 and max(lim_MappingTable[item]) > .2:
                mapping_dict[item] = str(lim_MappingTable[item].idxmax())
            else:
                mapping_dict[item] = ref + '_Unlabeled'

        #saving the new mappings and leiden clusters
        new_mapping = []
        for item in sm.sams[org].adata.obs['leiden_clusters']:
            new_mapping.append(mapping_dict[org + '_' + str(item)][3:])

        sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)] = new_mapping
        sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(reso)] = sm.sams[org].adata.obs['leiden_clusters']
        num_cell_types.append(sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)].nunique())
        t1 = time.time()
        print('finished mapping ' + str((reso/5)*(100/25)) + ' percent in ' + str(t1-t0) + ' seconds')

    #remove the temporary mappings and keep the actual mapping
    for i in r:
        if i != best_reso:
            print(i)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
        else:
            sm.sams[org].adata.obs[level + '_mapping'] = sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(i)]
            sm.sams[org].adata.obs[level + '_lc'] = sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(i)]
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
            sm.sams['mo'].adata.obs.to_csv('../../MO_metadata_soupx_cleaned_08262026_subclass_250_'+str(indexer)+'.csv')